# Lab 03: Tools & MCP

**Goal:** use Strands' built-in (vended) tools, connect to MCP servers, filter tool sets, and control how tools run.

Course module: `samples/03-mcp-tools`

Three sources of tools:
1. **Your functions** with `@tool` (Lab 01)
2. **Vended tools** that ship with the SDK: `strands.vended_tools` (`file_editor`, `shell`, `http_request`, `web_fetch`, …)
3. **MCP servers**, any [Model Context Protocol](https://modelcontextprotocol.io) server, local (stdio) or remote (HTTP)

In [ ]:
# Run this first in every lab: puts the repo root on sys.path and loads lab_config
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from lab_config import get_model, describe
print(describe())

## 1. Vended tools
The agent gets `file_editor` and works inside a scratch directory. `shell` is also available, but it runs real commands on your machine, so only enable it where you're comfortable with that.

In [ ]:
import os, pathlib
from strands import Agent
from strands.vended_tools import file_editor

workdir = pathlib.Path("workspace"); workdir.mkdir(exist_ok=True)
workdir_abs = workdir.resolve()

agent = Agent(
    model=get_model(),
    tools=[file_editor],
    system_prompt=f"You can create and edit files. Only work inside {workdir_abs}. Always use absolute paths.",
)
result = agent(f"Create {workdir_abs}/hello.py containing a function greet(name) that returns 'Hello, <name>!' "
      "plus a __main__ block that prints greet('Strands').")

In [ ]:
print((workdir / "hello.py").read_text())

## 2. Connect to a remote MCP server
This example uses the public, no-auth **AWS Knowledge MCP server** (AWS docs and best practices) over streamable HTTP. `MCPClient(url=...)` is the shorthand for HTTP servers. You can also pass a transport **factory** (`lambda: some_client(...)`), which is how stdio servers work (section 4).

Use the client as a context manager to inspect it directly.

In [ ]:
import logging
logging.getLogger("mcp").setLevel(logging.CRITICAL)

from strands.tools.mcp import MCPClient

KNOWLEDGE_URL = "https://knowledge-mcp.global.api.aws"

aws_knowledge = MCPClient(url=KNOWLEDGE_URL)

with aws_knowledge:
    tools = aws_knowledge.list_tools_sync()
    for t in tools:
        print(f"- {t.tool_name}: {(t.mcp_tool.description or '').splitlines()[0][:90]}")

## 3. Give an agent MCP tools and filter them
Pass the `MCPClient` straight into `tools=[...]`, and the agent manages the connection lifecycle. `tool_filters` limits which server tools the model sees. A smaller tool set uses fewer tokens and leads to fewer wrong tool choices. `prefix` namespaces tool names, which avoids collisions when you connect several servers.

In [ ]:
filtered = MCPClient(
    url=KNOWLEDGE_URL,
    prefix="kb",
    tool_filters={"allowed": [t.tool_name for t in tools if "search" in t.tool_name or "read" in t.tool_name]},
)

agent = Agent(
    model=get_model(),
    tools=[filtered],
    system_prompt="You are an AWS documentation assistant. Search the docs, then answer concisely and cite the doc URL.",
)
print("Agent tools:", agent.tool_names, "\n")
result = agent("What is the maximum timeout for an AWS Lambda function?")

## 4. Local MCP servers over stdio (optional)
Any MCP server you can launch as a process works the same way. For example, the AWS pricing server via `uvx` (needs AWS credentials):

```python
from mcp import stdio_client, StdioServerParameters

aws_pricing = MCPClient(
    lambda: stdio_client(StdioServerParameters(
        command="uvx",
        args=["awslabs.aws-pricing-mcp-server@latest"],
        env={"FASTMCP_LOG_LEVEL": "ERROR", "AWS_REGION": "us-east-1"},
    )),
    prefix="pricing",
)
agent = Agent(model=get_model(), tools=[aws_knowledge, aws_pricing])
```

## 5. Tool executors: concurrent vs sequential
By default Strands runs the tools from a single model turn **concurrently**. When order matters (step 2 depends on step 1), use `SequentialToolExecutor`.

In [ ]:
from strands import tool
from strands.tools.executors import SequentialToolExecutor

log = []

@tool
def step_one() -> str:
    """Perform the first step of the workflow."""
    log.append("one"); return "Step one complete: file created."

@tool
def step_two() -> str:
    """Perform the second step, which depends on step one."""
    log.append("two"); return "Step two complete: file processed."

agent = Agent(model=get_model(), tools=[step_one, step_two],
              tool_executor=SequentialToolExecutor(), callback_handler=None)
print(agent("Run step one and then step two, then report."))
print("execution order:", log)

## 🧪 Exercise
Combine tool sources: build one agent with `file_editor` and the filtered AWS Knowledge MCP client, and ask it to research a topic and write `workspace/notes.md`.

**Next:** [Lab 04: Callbacks & Streaming](04-callbacks-streaming.ipynb)

In [ ]:
# your code here